In [1]:
import nest_asyncio
nest_asyncio.apply()

In [2]:
import os
import asyncio
import logging

# from raganything import RAGAnything
from lightrag import LightRAG
from lightrag.llm.openai import openai_complete_if_cache, openai_embed
from lightrag.utils import EmbeddingFunc, setup_logger,logger ,wrap_embedding_func_with_attrs, set_verbose_debug

In [3]:
from dotenv import load_dotenv

# env
load_dotenv("../.env", override=True)

# Logger

setup_logger("lightrag", level="INFO")

# Config
WORKING_DIR = "./rag_storage"
if not os.path.exists(WORKING_DIR):
    os.mkdir(WORKING_DIR)

In [4]:
# LLM Model Function
async def llm_model_func(
    prompt, system_prompt=None, history_messages=[], keyword_extraction=False, **kwargs
) -> str:
    return await openai_complete_if_cache(
        os.getenv("LLM_MODEL"),
        prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=os.getenv("LLM_BINDING_API_KEY"),
        base_url=os.getenv("LLM_BINDING_HOST"),
        **kwargs,
    )

async def print_stream(stream):
    async for chunk in stream:
        if chunk:
            print(chunk, end="", flush=True)

In [5]:
async def initialize_rag():

    embedding_dim = int(os.getenv("EMBEDDING_DIM", 1536))
    token_limit = int(os.getenv("EMBEDDING_TOKEN_LIMIT", 8192))
    model_name = os.getenv("EMBEDDING_MODEL")

    # Step 1: define raw embedding function
    async def raw_embedding_func(texts):
        return await openai_embed.func(
            texts,
            api_key=os.getenv("LLM_BINDING_API_KEY"),
            base_url=os.getenv("LLM_BINDING_HOST"),
            model=model_name,
        )

    # Step 2: wrap embedding function
    embedding_func = wrap_embedding_func_with_attrs(
        embedding_dim=embedding_dim,
        max_token_size=token_limit,
        model_name=model_name,
    )(raw_embedding_func)

    # Step 3: initialize LightRAG
    rag = LightRAG(
        working_dir=WORKING_DIR,
        llm_model_func=llm_model_func,
        embedding_func=embedding_func,
        kv_storage="PGKVStorage",
        vector_storage="PGVectorStorage",
        graph_storage="Neo4JStorage",
    )

    await rag.initialize_storages()
    return rag


In [54]:
# Only run if want to Clear old data files
files_to_delete = [
    "graph_chunk_entity_relation.graphml",
    "kv_store_doc_status.json",
    "kv_store_full_docs.json",
    "kv_store_text_chunks.json",
    "vdb_chunks.json",
    "vdb_entities.json",
    "vdb_relationships.json",
]

for file in files_to_delete:
    file_path = os.path.join(WORKING_DIR, file)
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"Deleting old file:: {file_path}")

In [6]:
def configure_logging():
    """Configure logging for the application"""

    # Reset any existing handlers to ensure clean configuration
    for logger_name in ["uvicorn", "uvicorn.access", "uvicorn.error", "lightrag"]:
        logger_instance = logging.getLogger(logger_name)
        logger_instance.handlers = []
        logger_instance.filters = []

    # Get log directory path from environment variable or use current directory
    log_dir = os.getenv("LOG_DIR", os.getcwd())
    log_file_path = os.path.abspath(
        os.path.join(log_dir, "lightrag_docs_ingestion.log")
    )

    print(f"\nLightRAG docs ingestion log file: {log_file_path}\n")
    os.makedirs(os.path.dirname(log_dir), exist_ok=True)

    # Get log file max size and backup count from environment variables
    log_max_bytes = int(os.getenv("LOG_MAX_BYTES", 10485760))  # Default 10MB
    log_backup_count = int(os.getenv("LOG_BACKUP_COUNT", 5))  # Default 5 backups

    logging.config.dictConfig(
        {
            "version": 1,
            "disable_existing_loggers": False,
            "formatters": {
                "default": {
                    "format": "%(levelname)s: %(message)s",
                },
                "detailed": {
                    "format": "%(asctime)s - %(name)s - %(levelname)s - %(message)s",
                },
            },
            "handlers": {
                "console": {
                    "formatter": "default",
                    "class": "logging.StreamHandler",
                    "stream": "ext://sys.stderr",
                },
                "file": {
                    "formatter": "detailed",
                    "class": "logging.handlers.RotatingFileHandler",
                    "filename": log_file_path,
                    "maxBytes": log_max_bytes,
                    "backupCount": log_backup_count,
                    "encoding": "utf-8",
                },
            },
            "loggers": {
                "lightrag": {
                    "handlers": ["console", "file"],
                    "level": "INFO",
                    "propagate": False,
                },
            },
        }
    )

    # Set the logger level to INFO
    logger.setLevel(logging.INFO)
    # Enable verbose debug if needed
    set_verbose_debug(os.getenv("VERBOSE_DEBUG", "false").lower() == "true")

## Documents

In [7]:
skj_path = "../data/skj_documents"
permenpan_path = "../Data/Documents_pemerintah"
makalah_path = "../Data/makalah"

docs_paths = [skj_path, permenpan_path, makalah_path]

for path in docs_paths:
    if not os.path.exists(path):
        print(f"Warning: The path '{path}' does not exist. Please check the path and try again.")
    else:
        print(f"The path '{path}' exists and is ready for document ingestion.")

The path '../data/skj_documents' exists and is ready for document ingestion.
The path '../Data/Documents_pemerintah' exists and is ready for document ingestion.


#### Reader

In [8]:
# Used
import zipfile
from xml.etree import ElementTree as ET
from pathlib import Path
import pdfplumber

NS = {'w': 'http://schemas.openxmlformats.org/wordprocessingml/2006/main'}
VMERGE = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val'

def _read_docx(filepath):
    with zipfile.ZipFile(filepath) as z:
        root = ET.fromstring(z.read('word/document.xml'))
    lines = []
    for child in root.find('.//w:body', NS):
        tag = child.tag.split('}')[-1]
        if tag == 'p':
            text = ''.join(
                r.find('w:t', NS).text for r in child.findall('.//w:r', NS)
                if r.find('w:t', NS) is not None and r.find('w:t', NS).text
            ).strip()
            if text:
                lines.append(text)
        elif tag == 'tbl':
            for row in child.findall('w:tr', NS):
                cells = []
                for cell in row.findall('w:tc', NS):
                    vm = cell.find('.//w:vMerge', NS)
                    if vm is not None and vm.get(VMERGE) != 'restart':
                        continue
                    text = ''.join(
                        r.find('w:t', NS).text
                        for p in cell.findall('.//w:p', NS)
                        for r in p.findall('.//w:r', NS)
                        if r.find('w:t', NS) is not None and r.find('w:t', NS).text
                    ).strip()
                    if text:
                        cells.append(text)
                if cells:
                    lines.append(' | '.join(cells))
    return '\n'.join(lines)

def _read_pdf(filepath):
    with pdfplumber.open(filepath) as pdf:
        return '\n'.join(
            page.extract_text() for page in pdf.pages
            if page.extract_text()
        )

def read_folder(folder_path):
    results = []
    for path in Path(folder_path).rglob('*'):
        suffix = path.suffix.lower()
        if suffix == '.docx':
            text = _read_docx(path)
        elif suffix == '.pdf':
            text = _read_pdf(path)
        else:
            continue
        results.append({'filename': path.name, 'text': text})
        print(f"✓ {path.name} ({len(text)} karakter)")
    return results


### Ingestion skj

In [10]:
# Skj
print("Reading SKJ documents")
teks_skj_full = read_folder("../Data/skj_documents")
# Peraturan BPOM
print("Reading Peraturan BPOM documents")
teks_peraturan_bpom_full = read_folder("../Data/Documents_pemerintah")
#Renstra
print("Reading Renstra documents")
teks_renstra_full = read_folder("../data/renstra_bpom")

Reading SKJ documents
✓ 1. Form SKJ Pimpinan Tinggi Madya Sekretaris Utama (Level 5).docx (11372 karakter)
✓ 10. Draft_SKJ_Inspektur_alx_edit - Inspektur 2 v2.docx (11056 karakter)
✓ 11. Form SKJ JPT Pratama (Kepala Biro Umum)_rev1.docx (10188 karakter)
✓ 12. Form SKJ Administrator (Kabag PBJ)_rev1.docx (8224 karakter)
✓ 13. Form SKJ Administrator (Kabag PBMN)_rev1.docx (8572 karakter)
✓ 14. Form SKJ Administrator (Kabag Rumah Tangga)_rev1.docx (8781 karakter)
✓ 15. Form SKJ Administrator (Kabag PKP)_rev1.docx (8488 karakter)
✓ 16. Form SKJ Pengawas (Kasubag Protokol)_Rev.docx (7343 karakter)
✓ 17. Form SKJ Pengawas (Kasubag Kesekretariatan Kepala Badan)_Rev.docx (7601 karakter)
✓ 18. Form SKJ Pengawas (Kasubag Kesekretariatan Sekretaris Utama)_Rev.docx (7606 karakter)
✓ 2. Draft_SKJ_Inspektur_alx_edit - Inspektur Utama v2.docx (11431 karakter)
✓ 22. Form SKJ Pengawas (Kasubag Kesekretariatan Deputi Bidang Penindakan)_Rev.docx (7664 karakter)
✓ 23. Form SKJ Administrator (Kabag Tata Us

In [11]:
# Rag initialization
rag = await initialize_rag() 

# Test embedding function
test_text = ["This is a test string for embedding."]
embedding = await rag.embedding_func(test_text)
embedding_dim = embedding.shape[1]
print("\n=======================")
print("Test embedding function")
print("========================")
print(f"Test dict: {test_text}")
print(f"Detected embedding dimension: {embedding_dim}\n\n")

INFO: PostgreSQL table: LIGHTRAG_VDB_ENTITY_text_embedding_3_small_1536d
INFO: PostgreSQL table: LIGHTRAG_VDB_RELATION_text_embedding_3_small_1536d
INFO: PostgreSQL table: LIGHTRAG_VDB_CHUNKS_text_embedding_3_small_1536d
INFO: PostgreSQL, Retry config: attempts=10, backoff=3.0s, backoff_max=30.0s, pool_close_timeout=5.0s
INFO: PostgreSQL, VECTOR extension enabled
INFO: PostgreSQL, Connected to database at 127.0.0.1:5452/lightrag without SSL
INFO: chunk_id column already exists in LIGHTRAG_LLM_CACHE table
INFO: cache_type column already exists in LIGHTRAG_LLM_CACHE table
INFO: queryparam column already exists in LIGHTRAG_LLM_CACHE table
INFO: mode column does not exist in LIGHTRAG_LLM_CACHE table
INFO: chunks_list column already exists in LIGHTRAG_DOC_STATUS table
INFO: llm_cache_list column already exists in LIGHTRAG_DOC_CHUNKS table
INFO: track_id column already exists in LIGHTRAG_DOC_STATUS table
INFO: Index on track_id column already exists for LIGHTRAG_DOC_STATUS table
INFO: metada


Test embedding function
Test dict: ['This is a test string for embedding.']
Detected embedding dimension: 1536




## Documents Ingestion

In [12]:
# 1. Insert docs skj
await rag.ainsert(
    [doc["text"] for doc in teks_skj_full],
    file_paths=[doc["filename"] for doc in teks_skj_full]
)

INFO: Created 20 duplicate document records with track_id: insert_20260423_223044_d64813a5
ERROR: [] Missing required field for document doc-pre-Peraturan BPOM No. 28 Tahun 2025 Tentang Renstra BPOM 2025-2029.pdf: DocProcessingStatus.__init__() got an unexpected keyword argument 'multimodal_content'
INFO: Preserving 20 failed document entries for manual review
INFO: No valid documents to process after consistency check
INFO: Enqueued document processing pipeline stopped


'insert_20260423_223044_d64813a5'

In [13]:
# 2. Insert docs peraturan bpom
await rag.ainsert(
    [doc["text"] for doc in teks_peraturan_bpom_full],
    file_paths=[doc["filename"] for doc in teks_peraturan_bpom_full]
)

ERROR: [] Missing required field for document doc-pre-Peraturan BPOM No. 28 Tahun 2025 Tentang Renstra BPOM 2025-2029.pdf: DocProcessingStatus.__init__() got an unexpected keyword argument 'multimodal_content'
INFO: Preserving 20 failed document entries for manual review
INFO: Processing 2 document(s)
INFO: Extracting stage 1/2: Permenpan Nomor 15 Tahun 2019.pdf
INFO: Processing d-id: doc-a9837259597d035287cd9e209ca92808
INFO: Extracting stage 2/2: Peraturan BPOM Nomor 2 Tahun 2024.pdf
INFO: Processing d-id: doc-9c6c42a7ca9a3e7cf443791f17311176
INFO: LLM func: 4 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
INFO:  == LLM cache == saving: default:extract:5fe1b8b9390e131544d0cc474e9e0a7d
INFO:  == LLM cache == saving: default:extract:a0f07fb14652aa12539d4e50f60110ae
INFO:  == LLM cache == saving: default:extract:56a8a5457cea91c5456e3502dc0eec20
INFO:  == LLM cache == saving: default:extract:7e3a2a0928b5ea4c0b21cce3cd2005b0
INFO:  == LLM cache == saving:

'insert_20260423_223121_a7e7a38c'

ERROR: OpenAI API Rate Limit Error: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}
ERROR: OpenAI API Rate Limit Error: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}
ERROR: OpenAI API Rate Limit Error: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}
ERROR: OpenAI API Rate Limit Error: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}
ERROR: OpenAI API Rate Limit Error: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'code': 'too_many_requests'}}
ERROR: OpenAI API Rate Limit Error: Error code: 429 - {'error': {'message': 'Too Many Requests', 'type': 'too_many_requests', 'param': None, 'cod

In [18]:
# 3. Insert docs renstra
await rag.ainsert(
    [doc["text"] for doc in teks_renstra_full],
    file_paths=[doc["filename"] for doc in teks_renstra_full]
)

ERROR: [] Missing required field for document doc-pre-Peraturan BPOM No. 28 Tahun 2025 Tentang Renstra BPOM 2025-2029.pdf: DocProcessingStatus.__init__() got an unexpected keyword argument 'multimodal_content'
INFO: Preserving 20 failed document entries for manual review
INFO: Reset 1 documents from PROCESSING/FAILED to PENDING status
INFO: Processing 2 document(s)
INFO: Extracting stage 1/2: Permenpan Nomor 15 Tahun 2019.pdf
INFO: Processing d-id: doc-a9837259597d035287cd9e209ca92808
INFO: Extracting stage 2/2: Renstra Balai Besar POM di Jakarta Tahun 2025 2029.pdf
INFO: Processing d-id: doc-63cf0bdd1527dfd2149cdd456fa0040c
INFO: Chunk 1 of 10 extracted 23 Ent + 22 Rel chunk-6b708dafe0ef1a880a666b828ef56030
INFO: Chunk 2 of 10 extracted 28 Ent + 29 Rel chunk-96d74b7dea0b505de11bbd6a2ed00af4
INFO: Chunk 3 of 10 extracted 22 Ent + 32 Rel chunk-7ca09da38f9163acaf68ad0c030fcc66
INFO: Chunk 4 of 10 extracted 39 Ent + 38 Rel chunk-4fd99f316c33adc9407f570b5fe09f30
INFO: Chunk 5 of 10 extract

'insert_20260423_231057_895a1c37'

### Test query

In [19]:
from lightrag import QueryParam
result = await rag.aquery(
    "Apa saja yang menjadi persyaratan, kriteria dalam seleksi kompetensi bidang untuk pembuatan makalah? dan apa rencana strategis BPOM yang berkaitan dengan seleksi kompetensi bidang?",
    param=QueryParam(mode="mix")
)
print(result)

INFO:  == LLM cache == saving: mix:keywords:6509c68703ee2ebef129f146896f514d
INFO: Query nodes: Makalah seleksi kompetensi bidang, Penilaian makalah, Struktur makalah, Materi kompetensi bidang, Badan Pengawas Obat dan Makanan (BPOM), Seleksi kompetensi bidang BPOM (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 217 relations
INFO: Query edges: Persyaratan seleksi kompetensi bidang, Kriteria seleksi kompetensi bidang, Pembuatan makalah, Rencana strategis BPOM (top_k:40, cosine:0.2)
INFO: Global query: 46 entites, 40 relations
INFO: Naive query: 20 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 72 entities, 227 relations, 20 vector chunks
INFO: After truncation: 33 entities, 137 relations
INFO: Selecting 62 from 62 entity-related chunks by vector similarity
INFO: Find no additional relations-related chunks from 137 relations
INFO: Round-robin merged chunks: 82 -> 76 (deduplicated 6)
INFO: Final context: 33 entities, 137 relations, 13 chunks
INFO: Final chunks S+F/O: E

## Persyaratan dan Kriteria Seleksi Kompetensi Bidang untuk Pembuatan Makalah

Dalam pelaksanaan seleksi kompetensi bidang di lingkungan BPOM, salah satu metode yang digunakan adalah penulisan makalah. Persyaratan dan kriteria yang digunakan dalam penilaian penulisan makalah ini diatur secara rinci sebagai berikut:

### **Format dan Komponen Makalah**
Penulisan makalah secara umum harus mencakup:
1. **Topik**: Topik makalah ditetapkan oleh Ketua Panitia Seleksi bersama Panitia Seleksi.
2. **Format Makalah**:
   - **Pendahuluan**
   - **Analisis dan Sintesis**
   - **Rencana Strategis Jangka Panjang dan Menengah beserta Road Map**
   - **Plan of Action**
   - **Konklusi**

### **Kriteria Penilaian Penulisan Makalah**
Penilaian terhadap makalah dilakukan oleh anggota Panitia Seleksi dengan rincian kriteria sebagai berikut:
1. **Kesesuaian Judul dengan Tema**  
   - Menilai daya tangkap serta linearitas antara perintah yang diberikan dengan pelaksanaan tugas (nilai antara 40 s.d 100).
2. 

### Test retrieval konteks

In [25]:
from lightrag import QueryParam
# ── Prompt ────────────────────────────────────────────────────────────────────
QUERY_KEYWORDS = """
MATRIKS RINGKASAN ANALISIS SWOT
PEMETAAN VISI, MISI, TUJUAN, STRATEGI, SASARAN STRATEGIS DAN INDIKATOR
PEMETAAN ARAH KEBIJAKAN DAN STRATEGI BPOM
Form. 1 Penilaian Penulisan Makalah
sekertaris utama
Kesesuaian Judul dengan Tema
Kesesuaian Isi Makalah dengan Judul dan Tema
Sistematika Penulisan
Struktur Makalah (Pendahuluan, Analisis dan Sintesis, Rencana Strategis, Plan Of Action, Konklusi)
rencana strategis BPOM
SK 1 SK 2 SK 3 SK 4 SK 5 SK 6 SK 7 SK 8 SK 9 SK 10 SK 11 
MATRIKS RINGKASAN ANALISIS SWOT
PEMETAAN VISI, MISI, TUJUAN, STRATEGI, SASARAN STRATEGIS DAN INDIKATOR
PEMETAAN ARAH KEBIJAKAN DAN STRATEGI BPOM
"""


PROMPT_KONTEKS = """
Anda adalah asisten yang bertugas mengumpulkan konteks relevan untuk penilaian makalah dan ketajaman analisis makalah berdasarkan jabatan sekertaris utama dan Rencana Strategis BPOM.

Berdasarkan jabatan sekertaris utama, berikan ringkasan singkat tentang:
1. Deskripsi jabatan dan kompetensi yang diperlukan
2. Kriteria penilaian utama untuk posisi ini
3. Standar kualitas yang diharapkan dalam penulisan makalah

Berikan jawaban dalam format paragraf singkat, fokus pada poin-poin penting yang akan membantu dalam evaluasi makalah.
"""


try:
        context_response = await rag.aquery(
            query=QUERY_KEYWORDS,
            param=QueryParam(
                only_need_context=True, 
                mode="mix",
                user_prompt=PROMPT_KONTEKS,
                # enable_rerank=False,
                # max_token_for_context=2000,
            ),
        )
        context_response_result = context_response if context_response else "Konteks jabatan tidak ditemukan dalam knowledge base."
        print(context_response_result)
except Exception as e:
    print(f"Error retrieving context: {str(e)}")





INFO:  == LLM cache == saving: mix:keywords:caa9af57dd930858277825cda63119f3
INFO: Query nodes: BPOM, Sekertaris utama, SK 1, SK 2, SK 3, SK 4, SK 5, SK 6, SK 7, SK 8, SK 9, SK 10, SK 11, Pendahuluan, Analisis dan sintesis, Rencana strategis, Plan Of Action, Konklusi, Kesesuaian judul dengan tema, Kesesuaian isi makalah dengan judul dan tema (top_k:40, cosine:0.2)
INFO: Local query: 40 entites, 262 relations
INFO: Query edges: Matriks ringkasan analisis SWOT, Pemetaan visi misi tujuan strategi sasaran strategis dan indikator, Pemetaan arah kebijakan dan strategi BPOM, Rencana strategis BPOM, Penilaian penulisan makalah, Sistematika penulisan, Struktur makalah (top_k:40, cosine:0.2)
INFO: Global query: 53 entites, 40 relations
INFO: Naive query: 20 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 77 entities, 273 relations, 20 vector chunks
INFO: After truncation: 46 entities, 137 relations
INFO: Selecting 54 from 54 entity-related chunks by vector similarity
INFO: Find 2 ad


Knowledge Graph Data (Entity):

```json
{"entity": "BPOM", "type": "organisasi", "description": "BPOM (Badan Pengawas Obat dan Makanan) adalah lembaga pemerintah di Indonesia yang memegang peran utama dalam pengawasan dan pengaturan obat dan makanan. BPOM bertanggung jawab memastikan keamanan, mutu, dan kelayakan distribusi produk obat dan makanan yang beredar di masyarakat, serta memberikan jaminan perlindungan kepada masyarakat Indonesia terhadap risiko dari produk tersebut. Sebagai badan pengawas, BPOM menjalankan fungsi strategis dalam menjaga standar kualitas produk, dan menjadi institusi utama dalam pelaksanaan tata kelola pemerintahan yang baik, pengawasan internal, dan kontrol administrasi di bidang pengawasan obat dan makanan.\n\nSelain berfokus pada pengawasan produk, BPOM juga berperan dalam pengelolaan sumber daya manusia, khususnya dalam penerapan sistem merit dan pengelolaan barang milik negara. BPOM menjadi objek utama dalam pelaksanaan tugas-tugas keprotokolan, tata us